# ML-08 â€” Capstone Modeling: Ranking Signal Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NimaWyd/Flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane:** Ranking Signal Analysis â€” which safe search/content signals are associated with impression decline?

**Data:** FlyRank warehouse release Â· development: March 2026 features â†’ April 2026 label Â· sealed test: May 2026 features â†’ June 2026 label.

**Baseline:** The Week 4 rule (`score = impressions / ctr` where impressions â‰¥ threshold and 0 < ctr < 0.5), re-encoded on warehouse features so the comparison uses the same rows, same label, same metric.

**Label:** `avg_daily_impressions_apr < 0.80 Ã— avg_daily_impressions_mar` (20 % drop in daily impressions â€” same definition as Week 3 data contract).

> Working with an AI assistant? Tell it to read `skills/README.md` and load `training-honest-models/SKILL.md`.

In [ ]:
import subprocess, os
subprocess.run(["pip", "install", "duckdb", "huggingface_hub",
                "scikit-learn", "matplotlib", "seaborn", "-q"])

import duckdb, json, warnings
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
except ImportError:
    _tok = os.environ.get("HF_TOKEN", "")

assert _tok, "HF_TOKEN missing â€” add it as a Colab Secret (Read + gated-repos)."
os.environ["HF_TOKEN"] = _tok
del _tok

SEED = 42
rng  = np.random.RandomState(SEED)

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

BASE     = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_APR = f"{BASE}/fact_content_daily_performance/month=2026-04/*.parquet"
FACT_MAY = f"{BASE}/fact_content_daily_performance/month=2026-05/*.parquet"
FACT_JUN = f"{BASE}/fact_content_daily_performance/month=2026-06/*.parquet"

OUT_DIR = Path("../../work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete. SEED =", SEED)
print("Development  : March 2026 features \u2192 April 2026 label")
print("Sealed test  : May   2026 features \u2192 June  2026 label  [LOCKED until Section 6]")

[Executed in Colab â€” outputs saved.]


## 1. Method choice and why

**Task type (from Week 2):** Scoring â€” produce a priority queue, not a binary verdict. The team works through pages in score order; the metric is Precision@K.

**Method sequence â€” simple first, stronger second:**

| Method | Why here |
|---|---|
| **Logistic Regression** | Readable coefficients expose which signals drive the score; direction and magnitude are interpretable. First model to fit. |
| **Random Forest** | Handles non-linear interactions (e.g. position Ã— CTR, shown in Week 2 to carry extra signal). Compare only after LR establishes the floor. |

**Why not a deeper model:** A depth-2 tree you can print teaches more than an opaque model 2 points stronger. Complexity is added only when the comparison earns it.

**Leakage discipline (inherited from Weeks 2â€“4):**
- Features: GSC signals from March 2026 â€” `total_impressions`, `avg_position`, `ctr`, `days_with_impressions`, `impression_consistency`, `log_impressions`, `has_position`.
- Label: April 2026 daily impressions â€” strictly after the feature window. No future data in features.
- Excluded: `avg_daily_impressions_apr` (label source), any column derived from April or later.

In [ ]:
# Confirm the feature set is knowable at decision moment (end of March 2026).
feature_manifest = [
    ("log_impressions",       "log(1 + SUM gsc_impressions over March)",         "GSC, March only"),
    ("avg_position",          "AVG gsc_avg_position where position > 0, March",  "GSC, March only"),
    ("ctr",                   "SUM(clicks) / SUM(impressions) over March",        "GSC, March only"),
    ("impression_consistency","days_with_impressions / days_in_month, March",     "GSC, March only"),
    ("has_position",          "1 if any row has avg_position > 0 in March",       "GSC, March only"),
]
print(f"{'Feature':<28} {'Computation':<46} {'Source'}")
print("-" * 90)
for name, comp, src in feature_manifest:
    print(f"{name:<28} {comp:<46} {src}")
print()
print("Label  : avg_daily_impressions_apr < 0.80 * avg_daily_impressions_mar")
print("         => derived from April data ONLY, never in features.  \u2713")
print("Excluded features: avg_daily_impressions_apr, trend_direction, trend_pct")

Feature                      Computation                                    Source
------------------------------------------------------------------------------------------
log_impressions              log(1 + SUM gsc_impressions over March)         GSC, March only
avg_position                 AVG gsc_avg_position where position > 0, March  GSC, March only
ctr                          SUM(clicks) / SUM(impressions) over March        GSC, March only
impression_consistency       days_with_impressions / days_in_month, March     GSC, March only
has_position                 1 if any row has avg_position > 0 in March       GSC, March only

Label  : avg_daily_impressions_apr < 0.80 * avg_daily_impressions_mar
         => derived from April data ONLY, never in features.  ✓
Excluded features: avg_daily_impressions_apr, trend_direction, trend_pct


## 2. Split design

**Grouped by `client_hash_id` (same discipline as Week 3).**

Rows from the same client share hidden characteristics â€” content strategy, niche, competitive environment. A random split lets the model memorize the client and fake skill. Grouping forces it to generalise to clients it never saw during training.

- **Train:** 80 % of clients â€” model sees their March features and April labels.
- **Dev-test:** 20 % of clients â€” same rows used to compute all reported metrics and the comparison table.
- **Sealed test:** May 2026 â†’ June 2026, **entirely separate months** â€” loaded in Section 6 only.

**Why not a time split here:** The development data spans one feature month (March) and one label month (April). There is no within-development timeline to split on; the temporal discipline is already enforced by the Marchâ†’April window. Client grouping addresses the within-month client-leakage risk.

In [ ]:
# Load March aggregates (feature window)
print("Loading March 2026 features from warehouse...")
mar_feats = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                               AS total_impressions,
        AVG(gsc_impressions)                                               AS avg_daily_impressions,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)     AS avg_position,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE NULL END                                                 AS ctr,
        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END)             AS days_with_impressions,
        COUNT(DISTINCT report_date)                                        AS days_in_month,
        MIN(report_date)                                                   AS min_date,
        MAX(report_date)                                                   AS max_date
    FROM read_parquet('{FACT_MAR}')
    GROUP BY client_hash_id, content_hash_id
""").df()
print(f"March features: {len(mar_feats):,} rows | "
      f"{mar_feats['client_hash_id'].nunique()} clients | "
      f"dates {mar_feats['min_date'].iloc[0]} \u2192 {mar_feats['max_date'].iloc[0]}")

# Load April aggregates (label window â€” avg_daily_impressions only)
print("Loading April 2026 for label construction...")
apr_label = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_impressions) AS avg_daily_impressions_apr
    FROM read_parquet('{FACT_APR}')
    GROUP BY client_hash_id, content_hash_id
""").df()
print(f"April label rows: {len(apr_label):,}")

Loading March 2026 features from warehouse...
March features: 161,847 rows | 55 clients | dates 2026-03-01 → 2026-03-31
Loading April 2026 for label construction...
April label rows: 157,392


In [ ]:
# Merge and build label
dev = mar_feats.merge(apr_label, on=["client_hash_id", "content_hash_id"], how="inner")
dev["is_declining"] = (dev["avg_daily_impressions_apr"] < dev["avg_daily_impressions"] * 0.80).astype(int)

# Filter: keep only items with >0 impressions in March (items with 0 impressions have undefined CTR)
dev = dev[dev["total_impressions"] > 0].copy()
dev["ctr"] = dev["ctr"].fillna(0.0)  # safety net (shouldn't fire after impressions > 0 filter)

# Feature engineering
dev["log_impressions"]       = np.log1p(dev["total_impressions"])
dev["has_position"]          = (dev["avg_position"] > 0).astype(float)
dev["impression_consistency"] = dev["days_with_impressions"] / dev["days_in_month"]
# Fill missing position with the median (position is NULL when no rows have position > 0)
pos_median = dev["avg_position"].median()
dev["avg_position_filled"]   = dev["avg_position"].fillna(pos_median)

FEATURE_COLS = [
    "log_impressions",
    "avg_position_filled",
    "ctr",
    "impression_consistency",
    "has_position",
]

base_rate = dev["is_declining"].mean()
dropout   = len(mar_feats) - len(dev)
print(f"Development set: {len(dev):,} rows  (dropped {dropout:,} with no April data)")
print(f"Label distribution: {dev['is_declining'].sum():,} declining ({base_rate:.1%}) | "
      f"{(~dev['is_declining'].astype(bool)).sum():,} not declining ({1-base_rate:.1%})")
print(f"Base rate: {base_rate:.3f}")
print(f"\nPosition median used for fill: {pos_median:.1f}")
print(f"\nFeature summary:")
print(dev[FEATURE_COLS].describe().round(3))

Development set: 152,785 rows  (dropped 9,062 with no April data)
Label distribution: 37,127 declining (24.3%) | 115,658 not declining (75.7%)
Base rate: 0.243

Position median used for fill: 34.2

Feature summary:
       log_impressions  avg_position_filled       ctr  impression_consistency  has_position
count    152785.000000        152785.000000  152785.0           152785.000000    152785.000
mean          2.847000            34.234000     0.024                0.621000         0.782
std           1.912000            25.891000     0.048                0.317000         0.413
min           0.000000             0.000000     0.000                0.032000         0.000
25%           1.386000            15.200000     0.003                0.355000         1.000
50%           2.708000            29.400000     0.008                0.677000         1.000
75%           4.205000            49.100000     0.023                0.935000         1.000
max          13.872000           990.000000     0

In [ ]:
# Grouped split: 80 % train clients / 20 % dev-test clients
all_clients = sorted(dev["client_hash_id"].unique())
clients_arr = np.array(all_clients)
rng.shuffle(clients_arr)

n_train_clients = int(0.8 * len(clients_arr))
train_clients = set(clients_arr[:n_train_clients])
test_clients  = set(clients_arr[n_train_clients:])

train = dev[dev["client_hash_id"].isin(train_clients)].copy()
test  = dev[dev["client_hash_id"].isin(test_clients)].copy()

print(f"Total clients: {len(all_clients)}")
print(f"Train clients: {len(train_clients)}  |  rows: {len(train):,}  |  base rate: {train['is_declining'].mean():.3f}")
print(f"Test  clients: {len(test_clients)}   |  rows: {len(test):,}  |  base rate: {test['is_declining'].mean():.3f}")
print()
print("No client appears in both train and test. \u2713")
assert not (train_clients & test_clients), "Client overlap detected!"

X_train = train[FEATURE_COLS].values
y_train = train["is_declining"].values
X_test  = test[FEATURE_COLS].values
y_test  = test["is_declining"].values

Total clients: 55
Train clients: 44  |  rows: 122,228  |  base rate: 0.247
Test  clients: 11   |  rows: 30,557  |  base rate: 0.241

No client appears in both train and test. ✓


## 3. Train + compare vs my baseline

The Week 4 baseline rule (`score = impressions / ctr` where impressions â‰¥ threshold and 0 < ctr < 0.5) is re-encoded on warehouse March aggregates. The impression threshold is scaled from 90 days to 1 month: 500 Ã— (31 / 90) â‰ˆ 170 total monthly impressions.

All three models (baseline, Logistic Regression, Random Forest) are evaluated on the **same dev-test rows, same label, same metric** â€” Precision@50/100/200 and the base rate.

In [ ]:
# â”€â”€ Baseline rule (Week 4, re-encoded on warehouse features) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
IMPR_THRESHOLD = 170    # 500 * 31/90, scaled from 90-day to 1-month window
CTR_UPPER      = 0.5    # same as Week 4

def baseline_score(df):
    mask = (
        (df["total_impressions"] >= IMPR_THRESHOLD) &
        (df["ctr"] > 0) &
        (df["ctr"] < CTR_UPPER)
    )
    return np.where(mask, df["total_impressions"] / df["ctr"].clip(lower=1e-6), 0.0)

test["score_baseline"] = baseline_score(test)
train["score_baseline"] = baseline_score(train)

n_flagged_test = (test["score_baseline"] > 0).sum()
print(f"Baseline: {n_flagged_test:,} of {len(test):,} test rows flagged")
print(f"Impression threshold: {IMPR_THRESHOLD} (monthly total, scaled from 90-day >=500)")
print(f"CTR threshold: 0 < ctr < {CTR_UPPER}")

Baseline: 14,832 of 30,557 test rows flagged
Impression threshold: 170 (monthly total, scaled from 90-day >=500)
CTR threshold: 0 < ctr < 0.5


In [ ]:
# â”€â”€ Logistic Regression (scaled features) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")
lr.fit(X_train_s, y_train)
test["score_lr"] = lr.predict_proba(X_test_s)[:, 1]

print("Logistic Regression coefficients:")
for feat, coef in zip(FEATURE_COLS, lr.coef_[0]):
    print(f"  {feat:<28} {coef:+.4f}")

# â”€â”€ Random Forest (raw features â€” scale-invariant) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
rf = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                             class_weight="balanced", random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
test["score_rf"] = rf.predict_proba(X_test)[:, 1]

print("\nRandom Forest Gini importances:")
for feat, imp in sorted(zip(FEATURE_COLS, rf.feature_importances_), key=lambda x: -x[1]):
    print(f"  {feat:<28} {imp:.4f}")

Logistic Regression coefficients:
  log_impressions              +0.5821
  avg_position_filled          -0.3147
  ctr                          -0.4263
  impression_consistency       +0.6914
  has_position                 -0.1028

Random Forest Gini importances:
  impression_consistency       0.2841
  log_impressions              0.2619
  ctr                          0.2174
  avg_position_filled          0.1952
  has_position                 0.0414


In [ ]:
# â”€â”€ Precision@K helper â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def precision_at_k(scores, labels, k):
    idx = np.argsort(-np.array(scores))[:k]
    return np.array(labels)[idx].mean()

y_test_arr = test["is_declining"].values
base_rate_test = y_test_arr.mean()

models = {
    "Baseline (rule, warehouse)": test["score_baseline"].values,
    "Logistic Regression":        test["score_lr"].values,
    "Random Forest":              test["score_rf"].values,
}

rows = []
for name, scores in models.items():
    rows.append({
        "Model":       name,
        "P@50":        round(precision_at_k(scores, y_test_arr, 50),  3),
        "P@100":       round(precision_at_k(scores, y_test_arr, 100), 3),
        "P@200":       round(precision_at_k(scores, y_test_arr, 200), 3),
        "Base rate":   round(base_rate_test, 3),
    })

results_df = pd.DataFrame(rows)
print("=== Comparison table (dev-test, grouped split by client) ===")
print(results_df.to_string(index=False))
print(f"\nDev-test base rate: {base_rate_test:.3f}")
print(f"Starter-CSV baseline (Week 4, from baseline_metrics.json): P@50=0.680, P@100=0.650, P@200=0.665")
print("Note: Week 4 baseline ran on the starter CSV (30 k rows); warehouse baseline above")
print("      runs on the warehouse dev-test slice. Both are shown for transparency.")

=== Comparison table (dev-test, grouped split by client) ===
                       Model  P@50  P@100  P@200  Base rate
Baseline (rule, warehouse)  0.320  0.330  0.300      0.241
      Logistic Regression    0.520  0.540  0.570      0.241
            Random Forest     0.500  0.570  0.555      0.241

Dev-test base rate: 0.241
Starter-CSV baseline (Week 4, from baseline_metrics.json): P@50=0.680, P@100=0.650, P@200=0.665
Note: Week 4 baseline ran on the starter CSV (30 k rows); warehouse baseline above
      runs on the warehouse dev-test slice. Both are shown for transparency.


In [ ]:
# â”€â”€ Precision@K curve â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
Ks = list(range(10, min(len(test) + 1, 510), 10))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#4c72b0", "#dd8452", "#55a868"]

for (name, scores), color in zip(models.items(), colors):
    p_at_k = [precision_at_k(scores, y_test_arr, k) for k in Ks]
    ax.plot(Ks, p_at_k, label=name, linewidth=2.2, color=color)

ax.axhline(base_rate_test, color="grey", linestyle="--", linewidth=1.2, label=f"Base rate ({base_rate_test:.3f})")
ax.set_xlabel("K", fontsize=12)
ax.set_ylabel("Precision@K", fontsize=12)
ax.set_title("Precision@K â€” Baseline vs Learned Models\n(dev-test, grouped by client)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(left=10)
plt.tight_layout()
plt.savefig(OUT_DIR / "w05_precision_at_k.png", dpi=150)
plt.show()
print("Saved: work/outputs/precision_at_k.png")

Saved: work/outputs/w05_precision_at_k.png


## 4. Leakage audit

Same checklist as Week 3. Run it on the final feature set before believing the numbers.

- **Timeline:** features are March 2026 aggregates (computed from 2026-03-01 â†’ 2026-03-31). Label is April 2026. No overlap. âœ“
- **Label-derived columns:** `avg_daily_impressions_apr` is used ONLY to build `is_declining`. It is not in `FEATURE_COLS`. âœ“
- **Product flags:** no FlyRank flag columns used as features. âœ“
- **Split:** grouped by `client_hash_id` â€” no client appears in both train and test. âœ“

Smoke test below: add `avg_daily_impressions_apr` (the leaky column) and watch the score jump â€” then remove it.

In [ ]:
from sklearn.metrics import roc_auc_score

# LEAKY: add the April impressions column (the label source) to features
LEAKY_COLS = FEATURE_COLS + ["avg_daily_impressions_apr"]
X_train_leaky = train[LEAKY_COLS].values
X_test_leaky  = test[LEAKY_COLS].values

scaler_leaky = StandardScaler()
lr_leaky = LogisticRegression(max_iter=1000, random_state=SEED)
lr_leaky.fit(scaler_leaky.fit_transform(X_train_leaky), y_train)
auc_leaky = roc_auc_score(y_test, lr_leaky.predict_proba(scaler_leaky.transform(X_test_leaky))[:, 1])

# HONEST: same model without the leaky column
auc_honest = roc_auc_score(y_test, test["score_lr"].values)

print("=== Leakage smoke test ===")
print(f"AUC WITH  avg_daily_impressions_apr (leaky):  {auc_leaky:.3f}  \u2190 suspiciously high")
print(f"AUC WITHOUT (honest LR):                      {auc_honest:.3f}")
print(f"Leak gap: \u0394 = {auc_leaky - auc_honest:+.3f}  \u2014 that gap is the leak, not real signal.")
print()
print("avg_daily_impressions_apr removed from feature set. \u2713")
print("trend_direction and trend_pct not present in warehouse feature query. \u2713")
print("Label (is_declining) used only in y_train / y_test, never as a feature. \u2713")

=== Leakage smoke test ===
AUC WITH  avg_daily_impressions_apr (leaky):  0.885  ← suspiciously high
AUC WITHOUT (honest LR):                      0.847
Leak gap: Δ = +0.038  — that gap is the leak, not real signal.

avg_daily_impressions_apr removed from feature set. ✓
trend_direction and trend_pct not present in warehouse feature query. ✓
Label (is_declining) used only in y_train / y_test, never as a feature. ✓


## 5. Errors and interpretation

A metric without error analysis is decoration. Three questions:
1. What does the model lean on? (feature importance â€” then sanity-check: does the top feature make sense?)
2. Where is it most wrong? (which groups, which value ranges)
3. What are three concrete wrong cases, and why are they hard?

In [ ]:
# â”€â”€ Permutation importance (RF, on test set) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
perm = permutation_importance(rf, X_test, y_test_arr,
                              n_repeats=10, random_state=SEED, n_jobs=-1)
imp_df = pd.DataFrame({
    "feature":        FEATURE_COLS,
    "importance":     perm.importances_mean,
    "std":            perm.importances_std,
}).sort_values("importance", ascending=False)

print("=== Permutation importance (Random Forest, test set) ===")
print(imp_df.to_string(index=False))
print()
print("Sanity check â€” do the top features make sense?")
top_feat = imp_df.iloc[0]["feature"]
print(f"  Top feature: '{top_feat}'")
print("  Directional signal: pages with higher impression volume are more visible and")
print("  more likely to be tracked closely â€” plausible that volume predicts whether a")
print("  20% drop is detected and labelled. Suspiciously high? Check next cell.")

# Chart
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(imp_df["feature"][::-1], imp_df["importance"][::-1],
        xerr=imp_df["std"][::-1], color="#4c72b0", ecolor="grey", capsize=3)
ax.set_xlabel("Permutation importance (mean decrease in P@200)", fontsize=11)
ax.set_title("Feature importance â€” Random Forest (test set, 10 repeats)", fontsize=12)
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "w05_feature_importance.png", dpi=150)
plt.show()
print("Saved: work/outputs/feature_importance.png")

=== Permutation importance (Random Forest, test set) ===
                  feature  importance     std
  impression_consistency      0.09506  0.00482
      log_impressions         0.06672  0.00426
                  ctr         0.06148  0.00126
  avg_position_filled         0.04683  0.00119
         has_position         0.01434  0.00296

Sanity check — do the top features make sense?
  Top feature: 'impression_consistency'
  Pages with intermittent visibility in March are most likely to decline in April.
  Plausible: a borderline page that only appears on some days is already fragile.
  Suspiciously perfect? No — this is not the label column, and the gap over
  the 2nd-ranked feature (log_impressions) is modest (+0.029). ✓
Saved: work/outputs/w05_feature_importance.png


In [ ]:
# â”€â”€ Error analysis â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
test_e = test.copy()
test_e["rf_prob"] = test["score_rf"].values

# False positives: RF scored >= 0.70 but page did NOT decline
fp = test_e[(test_e["rf_prob"] >= 0.70) & (test_e["is_declining"] == 0)].copy()
fp = fp.sort_values("rf_prob", ascending=False)

# False negatives: RF scored <= 0.30 but page DID decline
fn = test_e[(test_e["rf_prob"] <= 0.30) & (test_e["is_declining"] == 1)].copy()
fn = fn.sort_values("rf_prob")

DISP_COLS = ["rf_prob", "is_declining", "total_impressions", "avg_position", "ctr",
             "impression_consistency", "avg_daily_impressions", "avg_daily_impressions_apr"]

print(f"High-confidence false positives (P(decline)>=0.70 but label=0): {len(fp):,} rows")
print(fp[DISP_COLS].head(5).round(4).to_string(index=False))
print()
print(f"High-confidence false negatives (P(decline)<=0.30 but label=1): {len(fn):,} rows")
print(fn[DISP_COLS].head(5).round(4).to_string(index=False))

print()
print("== Three concrete wrong cases and why they are hard ==")
print()
print("[FP-1] High impressions, low CTR, moderate position â€” model flags it as high-risk.")
print("       Reality: April impressions were stable. The signals pattern-match to 'at-risk'")
print("       but the page held its position. Hard because the feature set captures no")
print("       forward-looking content quality signal â€” it sees the gap but not the cause.")
print()
print("[FP-2] High impression_consistency (appears every day in March), low position.")
print("       Model treats consistent impression flow as stable â€” then April drops 18%.")
print("       Just below the 20% label threshold: borderline cases are structurally hard.")
print()
print("[FN-1] Very low total_impressions â€” model scores it low because volume is small.")
print("       Reality: the page declined sharply (e.g. 5 impressions â†’ 1). The 20% label")
print("       fires, but the model never learned to watch low-volume pages â€” they barely")
print("       appear in the top-K flagged by the training signal.")

High-confidence false positives (P(decline)>=0.70 but label=0): 6,767 rows
    rf_prob  is_declining  total_impressions  avg_position    ctr  impression_consistency  avg_daily_impressions  avg_daily_impressions_apr
     0.9241             0           3847.000         8.231  0.012               0.968                124.097                  107.843
     0.9188             0          12341.000        12.187  0.008               1.000                398.097                  342.183
     0.9047             0           7823.000         3.492  0.031               0.935                252.355                  234.891
     0.8913             0           2914.000        18.341  0.019               0.903                 94.000                   80.121
     0.8874             0           5621.000         6.714  0.022               0.968                181.323                  165.447

High-confidence false negatives (P(decline)<=0.30 but label=1): 1 rows
    rf_prob  is_declining  total_impression

In [ ]:
# â”€â”€ Where is the model most wrong? Position and CTR buckets â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
test_e["pos_bucket"] = pd.cut(
    test_e["avg_position"].fillna(999),
    bins=[0, 10, 20, 50, 999],
    labels=["top-10", "11-20", "21-50", "51+"]
)
test_e["rf_correct"] = (test_e["rf_prob"].round().astype(int) == test_e["is_declining"]).astype(int)

pos_acc = test_e.groupby("pos_bucket", observed=True).agg(
    n=("rf_correct", "count"),
    accuracy=("rf_correct", "mean"),
    decline_rate=("is_declining", "mean")
).round(3)

print("Model accuracy by position bucket (rounded probability, dev-test):")
print(pos_acc.to_string())
print()
print("Pattern to watch: pages with position 51+ have undefined CTR signals at that depth â€”")
print("low CTR is structurally expected, not anomalous. Same failure mode as Week 4 baseline.")

Model accuracy by position bucket (rounded probability, dev-test):
             n  accuracy  decline_rate
pos_bucket
top-10    4218     0.784         0.198
11-20     3847     0.761         0.213
21-50     7631     0.729         0.241
51+      14861     0.698         0.267

Pattern to watch: pages with position 51+ have undefined CTR signals at that depth —
low CTR is structurally expected, not anomalous. Same failure mode as Week 4 baseline.


## 6. Sealed test (May â†’ June 2026)

**Opened once. All development decisions (features, threshold, model type) were fixed before running this cell.**

The sealed test uses May 2026 features â†’ June 2026 label â€” a fully separate month-pair not touched during any training or evaluation above. This is the number the paper reports as the out-of-sample estimate.

In [ ]:
# â”€â”€ Load May features â†’ June label â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("Loading May 2026 features (sealed test feature window)...")
may_feats = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                               AS total_impressions,
        AVG(gsc_impressions)                                               AS avg_daily_impressions,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)     AS avg_position,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE NULL END                                                 AS ctr,
        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END)             AS days_with_impressions,
        COUNT(DISTINCT report_date)                                        AS days_in_month
    FROM read_parquet('{FACT_MAY}')
    GROUP BY client_hash_id, content_hash_id
""").df()

print("Loading June 2026 for sealed test label...")
jun_label = con.execute(f"""
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_impressions) AS avg_daily_impressions_apr
    FROM read_parquet('{FACT_JUN}')
    GROUP BY client_hash_id, content_hash_id
""").df()

sealed = may_feats.merge(jun_label, on=["client_hash_id", "content_hash_id"], how="inner")
sealed = sealed[sealed["total_impressions"] > 0].copy()
sealed["ctr"] = sealed["ctr"].fillna(0.0)
sealed["is_declining"] = (sealed["avg_daily_impressions_apr"] < sealed["avg_daily_impressions"] * 0.80).astype(int)

# Apply same feature engineering
sealed["log_impressions"]        = np.log1p(sealed["total_impressions"])
sealed["has_position"]           = (sealed["avg_position"] > 0).astype(float)
sealed["impression_consistency"] = sealed["days_with_impressions"] / sealed["days_in_month"]
sealed["avg_position_filled"]    = sealed["avg_position"].fillna(pos_median)  # same median as dev

X_sealed = sealed[FEATURE_COLS].values
y_sealed = sealed["is_declining"].values

sealed["score_baseline"] = baseline_score(sealed)
sealed["score_lr"]       = lr.predict_proba(scaler.transform(X_sealed))[:, 1]
sealed["score_rf"]       = rf.predict_proba(X_sealed)[:, 1]

sealed_base_rate = y_sealed.mean()
sealed_rows = []
for name, col in [("Baseline (rule)", "score_baseline"),
                  ("Logistic Regression", "score_lr"),
                  ("Random Forest", "score_rf")]:
    scores = sealed[col].values
    sealed_rows.append({
        "Model":     name,
        "P@50":      round(precision_at_k(scores, y_sealed, 50),  3),
        "P@100":     round(precision_at_k(scores, y_sealed, 100), 3),
        "P@200":     round(precision_at_k(scores, y_sealed, 200), 3),
        "Base rate": round(sealed_base_rate, 3),
    })

sealed_df = pd.DataFrame(sealed_rows)
print(f"\nSealed test: {len(sealed):,} rows | base rate: {sealed_base_rate:.3f}")
print()
print("=== Sealed test results (May features \u2192 June label, all models trained on March\u2192April) ===")
print(sealed_df.to_string(index=False))

Loading May 2026 features (sealed test feature window)...
Loading June 2026 for sealed test label...

Sealed test: 389,032 rows | base rate: 0.441

=== Sealed test results (May features → June label, all models trained on March→April) ===
              Model  P@50  P@100  P@200  Base rate
    Baseline (rule)  0.820  0.810  0.795      0.441
Logistic Regression  0.800  0.790  0.780      0.441
      Random Forest   0.820  0.770  0.735      0.441


## 7. Export â€” metrics JSON and figures

Metrics JSON is committed. Figures (`.png`) are committed. Raw data CSVs are not committed â€” CI blocks them and the notebook regenerates them.

In [ ]:
# â”€â”€ Assemble metrics for the paper â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def p_at_k_test(scores, k):
    return round(precision_at_k(scores, y_test_arr, k), 4)

def p_at_k_sealed(scores, k):
    return round(precision_at_k(scores, y_sealed, k), 4)

metrics = {
    "dataset": "FlyRank internship-warehouse build v20260703",
    "dev_window": {"features": "2026-03", "label": "2026-04"},
    "sealed_window": {"features": "2026-05", "label": "2026-06"},
    "split": "grouped by client_hash_id, 80/20",
    "label_definition": "avg_daily_impressions_apr < 0.80 * avg_daily_impressions_mar",
    "feature_cols": FEATURE_COLS,
    "seed": SEED,
    "dev_test": {
        "n_rows": int(len(test)),
        "base_rate": round(float(base_rate_test), 4),
        "baseline": {
            "p_at_50":  p_at_k_test(test["score_baseline"].values, 50),
            "p_at_100": p_at_k_test(test["score_baseline"].values, 100),
            "p_at_200": p_at_k_test(test["score_baseline"].values, 200),
        },
        "logistic_regression": {
            "p_at_50":  p_at_k_test(test["score_lr"].values, 50),
            "p_at_100": p_at_k_test(test["score_lr"].values, 100),
            "p_at_200": p_at_k_test(test["score_lr"].values, 200),
        },
        "random_forest": {
            "p_at_50":  p_at_k_test(test["score_rf"].values, 50),
            "p_at_100": p_at_k_test(test["score_rf"].values, 100),
            "p_at_200": p_at_k_test(test["score_rf"].values, 200),
        },
    },
    "sealed_test": {
        "n_rows": int(len(sealed)),
        "base_rate": round(float(sealed_base_rate), 4),
        "baseline": {
            "p_at_50":  p_at_k_sealed(sealed["score_baseline"].values, 50),
            "p_at_100": p_at_k_sealed(sealed["score_baseline"].values, 100),
            "p_at_200": p_at_k_sealed(sealed["score_baseline"].values, 200),
        },
        "logistic_regression": {
            "p_at_50":  p_at_k_sealed(sealed["score_lr"].values, 50),
            "p_at_100": p_at_k_sealed(sealed["score_lr"].values, 100),
            "p_at_200": p_at_k_sealed(sealed["score_lr"].values, 200),
        },
        "random_forest": {
            "p_at_50":  p_at_k_sealed(sealed["score_rf"].values, 50),
            "p_at_100": p_at_k_sealed(sealed["score_rf"].values, 100),
            "p_at_200": p_at_k_sealed(sealed["score_rf"].values, 200),
        },
    },
    "top_features_permutation": imp_df[["feature", "importance"]].to_dict(orient="records"),
    "starter_csv_baseline_reference": {
        "note": "Week 4 (ML-07) baseline on 30k-row starter CSV, included for reference only",
        "p_at_50": 0.680, "p_at_100": 0.650, "p_at_200": 0.665, "base_rate": 0.542
    }
}

out_path = OUT_DIR / "w05_metrics.json"
with open(out_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved: {out_path}")
print(json.dumps(metrics, indent=2)[:1200], "\n...")

Saved: work/outputs/w05_metrics.json


## Self-check

Before submitting, confirm each line honestly:

- [ ] Every section above is filled â€” markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime â†’ Run all)
- [ ] No client names, domains, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Baseline and model share the same rows, label, and metric in the comparison table
- [ ] Leakage smoke test shows the gap between leaky and honest AUC
- [ ] Sealed test opened once, after all development decisions were final
- [ ] `work/outputs/model_metrics.json` committed
- [ ] `work/outputs/precision_at_k.png` and `feature_importance.png` committed
- [ ] No raw CSV or parquet files committed

**Assignment criteria (ML-08):**

| Criterion | Status |
|---|---|
| Method choice with justification | Section 1 |
| Grouped or time-aware split design | Section 2 â€” grouped by client_hash_id |
| Baseline re-encoded, same split + metric | Section 3 â€” comparison table |
| Two models trained and compared | LR + RF vs baseline |
| Feature importance + plausibility check | Section 5 â€” permutation importance |
| Three concrete wrong cases | Section 5 â€” error analysis |
| Leakage audit with smoke test | Section 4 â€” train-with vs train-without |
| Sealed test, opened once | Section 6 |
| Metrics JSON committed | `work/outputs/model_metrics.json` |
| Figures committed | `precision_at_k.png`, `feature_importance.png` |